# Olist E-Commerce — Khám phá dữ liệu

Notebook này đọc trực tiếp từ `data/processed/olist.db` (do pipeline sinh ra) và
trình bày 5 phát hiện chính. Toàn bộ số liệu lấy từ các bảng `mart_*`, không tính lại bằng pandas.

**Chạy trước:** `python run_all.py`

> Báo cáo đầy đủ: `docs/findings.md`

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DB = Path('..', 'data', 'processed', 'olist.db').resolve()
assert DB.exists(), f'Chưa có database: {DB} — chạy python run_all.py trước'
conn = sqlite3.connect(DB)
q = lambda sql: pd.read_sql_query(sql, conn)
print(f'Database: {DB}  ({DB.stat().st_size / 1e6:.0f} MB)')

## 0. Quy mô dữ liệu

9 bảng gốc được nạp nguyên trạng vào `stg_*`, sau đó dựng thành 5 dimension + 3 fact.
Cột `rows` cho thấy độ lớn từng bảng.

In [ ]:
tables = q("""
    SELECT name FROM sqlite_master
    WHERE type = 'table'
      AND (name LIKE 'stg_%' OR name LIKE 'dim_%' OR name LIKE 'fact_%')
    ORDER BY name
""")
tables['rows'] = [q(f'SELECT COUNT(*) n FROM {t}')['n'][0] for t in tables['name']]
tables

## 1. Giao trễ phá huỷ điểm đánh giá

Đây là phát hiện mạnh nhất. Điểm review giảm **đơn điệu** theo mức độ trễ:
4,29 → 3,29 → 2,10 → 1,70 sao.

In [ ]:
d = q("""
    SELECT delivery_bucket, orders, avg_review_score, pct_low_review, avg_lead_time_days
    FROM mart_delivery_impact
    WHERE delivery_bucket != 'not_delivered'
    ORDER BY sort_order
""")
display(d)

ax = d.plot.bar(x='delivery_bucket', y='avg_review_score', legend=False,
                color='#1f4e79', figsize=(8, 3.6), ylim=(0, 5),
                title='Điểm review trung bình theo mức độ trễ')
ax.set_xlabel('Mức độ giao hàng')
ax.set_ylabel('Điểm review (1-5)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
q('SELECT * FROM mart_delivery_gap').T

## 2. Khách gần như không quay lại

**Lưu ý:** phải dùng `customer_unique_id`, không phải `customer_id`.
`customer_id` là duy nhất theo từng đơn nên dùng nó sẽ cho tỷ lệ quay lại = 0%.

In [ ]:
q('SELECT * FROM mart_customer_segments ORDER BY revenue DESC')

## 3. Seller: nhóm rủi ro nhỏ nhưng gây nhiều trễ

Mỗi điểm trên biểu đồ là một seller có ≥ 20 đơn. Kích thước điểm = số đơn.

In [ ]:
display(q('SELECT * FROM mart_seller_flags ORDER BY revenue DESC'))

s = q("""
    SELECT late_rate_pct, avg_review_score, orders, health_flag
    FROM mart_seller_scorecard WHERE orders >= 20
""")
colors = s['health_flag'].map({'healthy': '#2e7d32', 'watch': '#f0a020', 'at_risk': '#c00000'})
ax = s.plot.scatter(x='late_rate_pct', y='avg_review_score', s=s['orders'] / 4,
                    c=colors, alpha=0.6, figsize=(8, 4.2),
                    title='Seller: tỷ lệ trễ càng cao, điểm càng thấp')
ax.set_xlabel('Tỷ lệ giao trễ (%)')
ax.set_ylabel('Điểm review trung bình')
plt.tight_layout()
plt.show()

## 4. Địa lý: RJ trễ gấp 2,7 lần SP dù chỉ cách ~430 km

Đây là vấn đề vận hành theo tuyến, không phải khoảng cách.

In [ ]:
q("""
    SELECT state, orders, ROUND(revenue, 0) AS revenue, avg_lead_time_days,
           late_rate_pct, avg_review_score
    FROM mart_state_performance
    ORDER BY revenue DESC LIMIT 12
""")

## 5. Càng xa càng đắt, càng chậm, càng bị điểm thấp

In [ ]:
df = q("""
    SELECT distance_band, orders, avg_freight, avg_lead_time_days,
           late_rate_pct, avg_review_score
    FROM mart_freight_by_distance
    WHERE distance_band NOT LIKE '6_%'
    ORDER BY sort_order
""")
display(df)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
df.plot.bar(x='distance_band', y='avg_freight', legend=False, color='#1f4e79', ax=axes[0],
            title='Phí ship theo khoảng cách')
df.plot.bar(x='distance_band', y='avg_lead_time_days', legend=False, color='#c55a11', ax=axes[1],
            title='Thời gian giao theo khoảng cách')
for ax in axes:
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## Kết luận

| Phát hiện | Con số |
|---|---|
| Giao trễ mất | **2,02 sao** (4,29 → 2,27) |
| Khách trễ cho 1-2 sao | **62,41%** (so với 9,25% khi đúng hạn) |
| Khách quay lại | **3,04%** |
| Seller rủi ro | **67 seller**, 6,15% doanh thu, 18,06% trễ |
| Bang tệ nhất | **MA** (17,56% trễ), **RJ** (12,15%, 12.648 đơn) |

Đọc thêm: `docs/findings.md` (báo cáo đầy đủ) và `reports/figures/` (17 chart).

**Hạn chế:** dataset không có giá vốn, chi phí marketing hay traffic — nên không thể
kết luận về biên lợi nhuận, CAC/LTV hay tỷ lệ chuyển đổi.